In [1]:
!pip -q install transformers datasets accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 83.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incomp

In [2]:
import pandas as pd
import numpy as np
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [3]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample_submission = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

print(train_df.shape)
print(test_df.shape)
train_df.head()

(2000, 8)
(500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [4]:
train_df["text"] = (
    train_df["prompt"].fillna("") +
    " [SEP] A: " + train_df["A"].fillna("") +
    " [SEP] B: " + train_df["B"].fillna("") +
    " [SEP] C: " + train_df["C"].fillna("") +
    " [SEP] D: " + train_df["D"].fillna("") +
    " [SEP] E: " + train_df["E"].fillna("")
)

test_df["text"] = (
    test_df["prompt"].fillna("") +
    " [SEP] A: " + test_df["A"].fillna("") +
    " [SEP] B: " + test_df["B"].fillna("") +
    " [SEP] C: " + test_df["C"].fillna("") +
    " [SEP] D: " + test_df["D"].fillna("") +
    " [SEP] E: " + test_df["E"].fillna("")
)

train_df[["text", "answer"]].head()

,text,answer
0,Pick the best possible answer: What is Martin ...,B
1,What is accelerator-based light-ion fusion? [S...,A
2,Determine the correct option: What is the term...,C
3,Select the most accurate option: What is Marti...,B
4,Identify the correct statement: What is the co...,A


In [5]:
label_map = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

train_df["label"] = train_df["answer"].map(label_map)

train_df[["answer", "label"]].head()

,answer,label
0,B,1
1,A,0
2,C,2
3,B,1
4,A,0


In [6]:
from sklearn.model_selection import train_test_split

train_data, valid_data = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df["label"]
)

print(train_data.shape)
print(valid_data.shape)

(1600, 10)
(400, 10)


In [7]:
MODEL_NAME = "FacebookAI/roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
train_dataset = Dataset.from_pandas(
    train_data[["text", "label"]]
)

valid_dataset = Dataset.from_pandas(
    valid_data[["text", "label"]]
)

test_dataset = Dataset.from_pandas(
    test_df[["text"]]
)

In [9]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
valid_dataset = valid_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [10]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

valid_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask"]
)

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=5
)

print("Model loaded successfully!")

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully!


In [12]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",

    do_train=True,
    do_eval=True,

    eval_strategy="epoch",
    save_strategy="no",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=3,

    weight_decay=0.01,

    logging_steps=50,

    report_to="none",

    fp16=torch.cuda.is_available()
)

In [13]:
import numpy as np

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(logits, axis=-1)

    acc = (preds == labels).mean()

    return {"accuracy": acc}

In [14]:
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=valid_dataset,

    processing_class=tokenizer,

    compute_metrics=compute_metrics
)

In [15]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,2.970431,2.388458,0.540000
2,0.644145,0.182468,1.000000
3,0.062880,0.032042,1.000000


TrainOutput(global_step=300, training_loss=1.4823667152722677, metrics={'train_runtime': 156.7677, 'train_samples_per_second': 30.619, 'train_steps_per_second': 1.914, 'total_flos': 631483541913600.0, 'train_loss': 1.4823667152722677, 'epoch': 3.0})

In [16]:
results = trainer.evaluate()

print(results)

{'eval_loss': 0.03204205632209778, 'eval_accuracy': 1.0, 'eval_runtime': 4.1635, 'eval_samples_per_second': 96.072, 'eval_steps_per_second': 6.004, 'epoch': 3.0}


In [17]:
# Predict on test set
test_predictions = trainer.predict(test_dataset)

# Extract logits
test_logits = test_predictions.predictions

print("Test logits shape:", test_logits.shape)

Test logits shape: (500, 5)


In [18]:
import numpy as np

# Convert logits to probabilities
probabilities = torch.softmax(
    torch.tensor(test_logits),
    dim=1
).numpy()

# Label mapping
labels = np.array(["A", "B", "C", "D", "E"])

# Top-3 predictions
top3 = np.argsort(probabilities, axis=1)[:, -3:][:, ::-1]

prediction_strings = [
    " ".join(labels[idx])
    for idx in top3
]

print(prediction_strings[:5])

['A D C', 'B D A', 'B C D', 'E C D', 'C D A']


In [19]:
submission = sample_submission.copy()

submission["Prediction"] = prediction_strings

submission.to_csv("submission.csv", index=False)

submission.head()

,ID,Prediction
0,1,A D C
1,2,B D A
2,3,B C D
3,4,E C D
4,5,C D A


In [20]:
import os

print(submission.shape)
print(sample_submission.shape)

print(submission.columns)

print(os.listdir("/kaggle/working"))

(500, 2)
(500, 2)
Index(['ID', 'Prediction'], dtype='object')
['submission.csv', 'results', '__notebook__.ipynb']
